#  PIIMiddleware中间件
件用于检测和处理对话中的个人身份信息（Personally Identifiable Information，PII），支持自定义处理策略。
## 参数说明
### 参数1： pii_type 检测的PII数据类型
- 可选值：
  - 'EMAIL' 邮箱
  - 'PHONE' 手机号
  - 'NAME' 姓名
  - 'ADDRESS' 地址
  - 'ID_NUMBER' 身份证号

### 参数2： strategy —处理PII信息的策略
- redact ：将检测到的PII信息用字符串 [REDACTED_[PII_TYPE]] 替换，其中的 PII_TYPE是上面提到的具体类型，比如 [REDACTED_EMAIL] 、 [REDACTED_CREDIT_CARD] 这样的标签。完全 “擦除/隐藏” 真实内容。适合日志清洗、合规需求、公开输出时隐藏敏感内容。
- mask ：用 *** 将PII信息的前面一部分信息遮蔽。比如信用卡号可能变成 ****-****-****-1234 （只保留最后几位/部分可见），邮箱可能保留域名部分 + 隐藏用户名的一部分等 — 既隐藏大部分敏感信息，又保留了一点“可辨识性”（比如账号后四位、域名等），适合用户服务界面/ 前端显示 / 需要部分可识别但不泄露完整敏感内容的场景。
- hash ：用检测到的PII信息的 哈希值 替代原值。比如 <email_hash:a1b2c3d4> 。适合analytics、调试 (debug)、统计分析、匿名追踪等场景。
- block ：如果检测到PII信息， 直接抛出异常 。适合对隐私要求极高、绝不允许泄露任何敏感信息的场景
- 
### 参数3：detector —自定义 PII检测函数 或者 正则表达式
如果没有提供则使用内置的检测函数

### 参数4：apply_to_input —是否在调用模型前检测
默认为True。

### 参数5：apply_to_output —是否在模型调用后检测
默认为False。

### 参数6：apply_to_tool_results —是否在工具调用后检测其输出
默认为False。

In [ ]:
# 1、模型的初始化
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen

custom_profile = {
"max_input_tokens": 128_000 # 最大上下文长度
}

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    profile=custom_profile, # 手动添加的配置项
)


In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage
agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware("url", strategy="hash", apply_to_input=True),
        PIIMiddleware("mac_address", strategy="mask", apply_to_input=True),
        PIIMiddleware("ip", strategy="block", apply_to_input=True),
    ]
)

response = agent.invoke({"messages": [HumanMessage("""
帮我向 156168188@qq.com 发送一封邮件
同时查看银行卡号： 5105-1051-0510-5100 的余额
访问 https://localhost:12345
确认这是不是 MAC地址： 11-11-11-11-11-11
""")]
})
for msg in response["messages"]:
    msg.pretty_print()
    try:
        response1 = agent.invoke({"messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]})
    except Exception as e:
        print('=' * 30, '-> 抛异常 <-', '=' * 30)
        print(f"检测到IP，抛出异常：{e}")

================================ Human Message =================================


帮我向 [REDACTED_EMAIL] 发送一封邮件
同时查看银行卡号： ****-****-****-5100 的余额
访问 <url_hash:dd5fc2a9>
确认这是不是 MAC地址： **-**-**-**-**-11

============================== -> 抛异常 <- ==============================
检测到IP，抛出异常：Detected 1 instance(s) of ip in text content
================================== Ai Message ==================================

作为一个人工智能助手，我无法直接操作外部系统或访问您的个人账户。以下是对您四项请求的逐一说明与安全建议：

📧 **发送邮件至 `[REDACTED_EMAIL]`**  
我无法代您发送邮箱，但您可以：
- 在本地邮箱客户端（Outlook、Foxmail、Apple Mail 等）或网页邮箱中手动发送；
- 如需我帮您撰写邮件正文，请告诉我收件人用途、主题及核心内容，我可为您生成正式模板。

🏦 **查询银行卡号 `****-****-****-5100` 余额**  
出于资金安全与合规要求，我**无法连接任何银行系统或查询账户信息**。请务必通过以下方式自查：
- 登录发卡行官方手机银行 App 或网银；
- 拨打银行客服专线（卡面背面号码），完成身份验证后查询；
- ⚠️ 请勿在非必要场景输入完整卡号、CVV 或短信验证码。

🔗 **访问 `<url_hash:dd5fc2a9>`**  
该链接为哈希加密格式，我无法解析或跳转。建议您：
- 将完整 URL 粘贴至浏览器地址栏打开；
- 使用前核对域名是否为官网，警惕钓鱼网站；
- 如怀疑链接来源不安全，可使用 [VirusTotal](https://www.virustotal.com) 进行在线检测。

🌐 **确认 `**-**-**-**-**-11` 是否为 MAC 地址**  
MA

In [12]:
import re
# 自定义检测函数
def detect_phone_number(content: str):
    return [
        {
            "value": m.group(0), # 提取出具体匹配到的 11 位数字文本（例如"13800138000"）
            "start": m.start(), # 这段数字在原文本中的“起始索引位置”（从 0 开始算）
            "end": m.end(), # 这段数字在原文本中的“结束索引位置”
            "type": "phone_number"
        } for m in re.finditer(r"[0-9]{11}", content)]

In [4]:
text = "尚硅谷的电话是13812345678，康师傅的电话是13987654321。"
result = detect_phone_number(text)
print(result)


[{'text': '13812345678', 'start': 7, 'end': 18}, {'text': '13987654321', 'start': 26, 'end': 37}]


In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True,detector=r"sk-[a-zA-Z0-9]+"),
        PIIMiddleware("phone_number", strategy="mask", apply_to_input=True,detector=detect_phone_number)
    ])
    
response = agent.invoke({"messages": [HumanMessage(
    """
        这是不是有效的 API_KEY：sk-awef23AFEfaafaefa
        帮我给这个号码打电话： 12345612345
        访问 https://localhost:12345
    """
    )]
})
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


        这是不是有效的 API_KEY：<api_key_hash:6c678cc0>
        帮我给这个号码打电话： ****2345
        访问 https://localhost:12345
    
================================== Ai Message ==================================

您好！针对您提到的三点，我逐一为您说明：

🔑 **关于 `API_KEY`**  
`<api_key_hash:6c678cc0>` 是一个典型的**脱敏/占位符格式**，并非真实有效的 API 密钥。实际的有效密钥通常是一串较长的字母、数字及符号组合（如 `sk-...`、`ghp_...` 等）。⚠️ **安全提醒**：任何真实密钥都应在环境变量或密钥管理系统中存储，切勿在对话、代码注释或公开文档中明文传输。

📞 **关于拨打电话**  
我无法执行外部硬件操作（如拨打电话），且 `****2345` 是部分隐藏的号码，缺少归属地或完整前缀，也无法通过任何正常渠道拨通。请您使用手机通讯录、运营商官方 App 或电脑软电话，填入完整号码后进行呼叫。

🌐 **关于访问 `https://localhost:12345`**  
该地址指向您**本机（localhost）**运行的服务。作为云端 AI 模型，我没有任何权限访问您的设备网络，更无法打开本地端口。您可以：
- 直接在浏览器中打开该地址
- 使用终端命令：`curl -k https://localhost:12345`
- 使用 Postman / Apifox 等工具发起请求

💡 如果您需要帮助：检查本地服务是否正常运行、配置安全的密钥管理方式、或编写调用接口的代码示例，请随时告诉我具体场景，我会为您提供详细方案。
